In [125]:
import torch
import torch.nn as nn

In [126]:

inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

### Simple self attention

In [127]:
cntx_vector=torch.zeros(inputs.shape[0],inputs.shape[1])
print(cntx_vector)        

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [128]:
for i,inp in enumerate(inputs):
    for j,inpi in enumerate(torch.softmax(inp @ torch.transpose(inputs,1,0),dim=0)):
        cntx_vector[i] +=inpi*inputs[j]

In [129]:
print(cntx_vector)

tensor([[0.442, 0.593, 0.579],
        [0.442, 0.651, 0.568],
        [0.443, 0.650, 0.567],
        [0.430, 0.630, 0.551],
        [0.467, 0.591, 0.527],
        [0.418, 0.650, 0.565]])


### Self attention using learnable weights

In [130]:
d_out=8
d_model=inputs.shape[1]
torch.manual_seed(123)
wq=nn.Linear(d_model,d_out,bias=False)
wk=nn.Linear(d_model,d_out,bias=False)
wv=nn.Linear(d_model,d_out,bias=False)

In [131]:
# Query, Key and Value
Q=wq(inputs)
K=wk(inputs)
V=wv(inputs)
#Attention Score
att_score=Q@(K.T)
#attention weight
att_weight=torch.softmax(att_score/d_out**0.5,dim=-1)
print(att_weight.shape)
#Context vector
context_vactor=att_weight@V
print(context_vactor)

torch.Size([6, 6])
tensor([[ 0.540,  0.252, -0.536, -0.011,  0.034, -0.293, -0.392,  0.602],
        [ 0.537,  0.253, -0.537, -0.018,  0.027, -0.289, -0.396,  0.601],
        [ 0.537,  0.253, -0.537, -0.017,  0.027, -0.289, -0.396,  0.601],
        [ 0.536,  0.252, -0.535, -0.017,  0.027, -0.289, -0.395,  0.600],
        [ 0.538,  0.252, -0.535, -0.015,  0.028, -0.290, -0.394,  0.600],
        [ 0.536,  0.252, -0.536, -0.018,  0.026, -0.288, -0.395,  0.600]],
       grad_fn=<MmBackward0>)


### Casual attention mask

In [132]:
wq=nn.Linear(d_model,d_out,bias=False)
wk=nn.Linear(d_model,d_out,bias=False)
wv=nn.Linear(d_model,d_out,bias=False)
# Query, Key and Value
Q=wq(inputs)
K=wk(inputs)
V=wv(inputs)
#Attention Score
att_score=Q@(K.T)
casual_mask=torch.tril(torch.ones(att_score.shape))
casual_mask=casual_mask.masked_fill(casual_mask==0,-torch.inf)
print(casual_mask)
print(att_score)
masked_att_score=att_score + casual_mask
print(masked_att_score)
att_weight=torch.softmax(masked_att_score/d_out**0.5,dim=-1)
print(att_weight)
#attention weight
#Context vector
context_vactor=att_weight@V
print(context_vactor)

tensor([[1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1.]])
tensor([[-0.080, -0.202, -0.204, -0.109, -0.182, -0.097],
        [-0.759, -0.840, -0.839, -0.412, -0.575, -0.460],
        [-0.745, -0.825, -0.824, -0.403, -0.571, -0.447],
        [-0.502, -0.539, -0.536, -0.265, -0.343, -0.309],
        [-0.297, -0.324, -0.330, -0.140, -0.343, -0.096],
        [-0.653, -0.709, -0.703, -0.359, -0.398, -0.444]],
       grad_fn=<MmBackward0>)
tensor([[0.920,  -inf,  -inf,  -inf,  -inf,  -inf],
        [0.241, 0.160,  -inf,  -inf,  -inf,  -inf],
        [0.255, 0.175, 0.176,  -inf,  -inf,  -inf],
        [0.498, 0.461, 0.464, 0.735,  -inf,  -inf],
        [0.703, 0.676, 0.670, 0.860, 0.657,  -inf],
        [0.347, 0.291, 0.297, 0.641, 0.602, 0.556]], grad_fn=<AddBackward0>)
tensor([[1.000, 0.000, 0.000, 0.000, 0.000, 0.000

#### Multihead Attention using weight split

In [133]:
class causal_attention(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias):
        super().__init__()
        #torch.manual_seed(123)
        self.W_query=nn.Linear(d_in,d_out,qkv_bias)
        self.W_key=nn.Linear(d_in,d_out,qkv_bias)
        self.W_value=nn.Linear(d_in,d_out,qkv_bias)
    def forward(self,x):
        query=self.W_query(x)
        key=self.W_key(x)
        value=self.W_value(x)
        #Attention Score
        attention_score=query @ (key.T)
        casual_mask=torch.tril(torch.ones(attention_score.shape))
        casual_att_score=casual_mask.masked_fill(casual_mask==0,-torch.inf)
        #attention weights
        attention_weight=torch.softmax(casual_att_score/(d_out**0.5),dim=-1)
        #context vector
        context_vector=attention_weight @ value
        return context_vector       

In [134]:
self_att=causal_attention(inputs.shape[1],8,False)
print(self_att(inputs))

tensor([[-0.235, -0.423,  0.088, -0.139, -0.182,  0.322,  0.134, -0.247],
        [-0.003, -0.329,  0.184, -0.048, -0.209,  0.250,  0.077, -0.198],
        [ 0.075, -0.299,  0.220, -0.017, -0.221,  0.224,  0.055, -0.185],
        [ 0.099, -0.243,  0.198, -0.002, -0.189,  0.187,  0.044, -0.147],
        [ 0.127, -0.234,  0.238,  0.009, -0.220,  0.155,  0.012, -0.168],
        [ 0.129, -0.207,  0.205,  0.013, -0.185,  0.152,  0.023, -0.134]],
       grad_fn=<MmBackward0>)


### Simple multihead attention

In [135]:
class SimpleMultiheadAttention(nn.Module):
    def __init__(self,d_in,d_out,qkv_bias,n_heads):
        super().__init__()
        self.heads=nn.ModuleList([causal_attention(d_in,d_out,qkv_bias) for _ in range(n_heads)])
    def forward(self,x):
        return torch.concat([head(x) for head in self.heads],dim=-1)

In [136]:
torch.manual_seed(123)
simpmultihead=SimpleMultiheadAttention(inputs.shape[1],8,False,2)
#Multihead attention context vector
torch.set_printoptions(sci_mode=False, precision=4)
print(simpmultihead(inputs))

tensor([[     0.4566,      0.2729,     -0.4303,     -0.1992,     -0.3749,
             -0.1810,     -0.5684,      0.5063,      0.4675,     -0.2793,
              0.2869,      0.2964,     -0.1885,     -0.0813,      0.0005,
              0.4467],
        [     0.5909,      0.3038,     -0.5869,     -0.1034,     -0.1349,
             -0.2898,     -0.5345,      0.6644,      0.4652,     -0.0636,
              0.3650,      0.2223,     -0.3285,     -0.0185,     -0.0595,
              0.4312],
        [     0.6355,      0.3125,     -0.6337,     -0.0653,     -0.0539,
             -0.3272,     -0.5200,      0.7132,      0.4580,      0.0049,
              0.3902,      0.2012,     -0.3713,     -0.0022,     -0.0779,
              0.4199],
        [     0.5730,      0.2794,     -0.5866,     -0.0554,     -0.0112,
             -0.2982,     -0.4530,      0.6524,      0.4113,      0.0460,
              0.3527,      0.1547,     -0.3529,      0.0184,     -0.0805,
              0.3754],
        [     0.5594